In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from sentence_transformers import CrossEncoder
import requests
from bs4 import BeautifulSoup
import gradio as gr
import ast, os

MODEL = "gpt-4.1-nano"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

In [2]:
HEADERS = {"User-Agent": "Mozilla/5.0"}

def scrape(url):
    try:
        r = requests.get(url, headers=HEADERS, timeout=10)
        soup = BeautifulSoup(r.content, "html.parser")
        for tag in soup(["script", "style", "nav", "footer"]):
            tag.decompose()
        return Document(page_content=soup.get_text("\n", strip=True), metadata={"source": url})
    except Exception as e:
        print(f"failed {url} : {e}")
        return None

urls = [
    "https://www.cars24.com/",
    "https://www.cars24.com/buy-used-cars/",
    "https://www.cars24.com/sell-used-cars/",
    "https://www.cars24.com/car-loans/",
]

docs = [d for d in [scrape(u) for u in urls] if d]
print(f"got {len(docs)} pages")

got 4 pages


In [ ]:
import shutil

# delete empty db
if os.path.exists(DB_NAME):
    shutil.rmtree(DB_NAME)
    print("deleted old db")

# scrape
docs = [d for d in [scrape(u) for u in urls] if d]
print(f"scraped {len(docs)} pages")

# chunk + build
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(docs)
print(f"got {len(chunks)} chunks")

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")  # ✅ added

vectorstore = Chroma.from_documents(chunks, embeddings, persist_directory=DB_NAME)
print(f"✅ db built with {vectorstore._collection.count()} chunks")

retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

deleted old db
scraped 4 pages
got 31 chunks


NameError: name 'embeddings' is not defined

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

if os.path.exists(DB_NAME) and os.listdir(DB_NAME):
    vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)
    print("loaded existing db")
else:
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    chunks = splitter.split_documents(docs)
    vectorstore = Chroma.from_documents(chunks, embeddings, persist_directory=DB_NAME)
    print(f"built db with {len(chunks)} chunks")

loaded existing db


In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})
llm = ChatOpenAI(model_name=MODEL, temperature=0)
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(question, docs, top_n=3):
    if not docs: return []
    scores = reranker.predict([(question, d.page_content) for d in docs])
    return [d for _, d in sorted(zip(scores, docs), reverse=True)][:top_n]

In [ ]:
def rewrite(q):
    r = llm.invoke([
        SystemMessage(content="rewrite this question to search cars24.com better. return only the query."),
        HumanMessage(content=q)
    ])
    print(f"rewritten: {r.content}")
    return r.content.strip()

def expand(q):
    r = llm.invoke([
        SystemMessage(content='give 3 search queries for this question. return only a python list like ["q1","q2","q3"]'),
        HumanMessage(content=q)
    ])
    try: return ast.literal_eval(r.content.strip())
    except: return [q]

def retrieve(question):
    rq = rewrite(question)
    queries = expand(rq) + [rq]
    
    all_docs = []
    for q in queries:
        all_docs.extend(retriever.invoke(q))
    
    seen, unique = set(), []
    for d in all_docs:
        if d.page_content not in seen:
            seen.add(d.page_content)
            unique.append(d)
    
    print(f"got {len(unique)} chunks")
    return rerank(rq, unique)

In [ ]:
PROMPT = """
You are a helpful assistant for CARS24 — India's leading used car platform.
Help users with buying, selling, loans, and services like FASTag and RTO.
If you don't know say: contact care@cars24.com

context:
{context}
"""

def answer(question, history):
    docs = retrieve(question)
    if not docs:
        return "couldn't find anything, try care@cars24.com"
    
    context = "\n\n".join(d.page_content for d in docs)
    msgs = [SystemMessage(content=PROMPT.format(context=context))]
    
    for h, a in history:
        msgs += [HumanMessage(content=h), AIMessage(content=a)]
    msgs.append(HumanMessage(content=question))
    
    return llm.invoke(msgs).content

gr.ChatInterface(fn=answer, title="cars24 bot").launch()

c:\Users\tbfro\projects\llm_engineering\.venv\Lib\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


rewritten: used cars in India site:cars24.com
got 0 chunks
rewritten: Search for used cars on cars24.com
got 0 chunks
rewritten: Why is Cars24.com the best platform to buy or sell used cars?
got 0 chunks


In [ ]:
print(vectorstore._collection.count())

0
